In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

# Make sure we're in the right directory
os.chdir('..')

# Load the saved segments and labels
X = np.load('data/processed/X_segments.npy')
y = np.load('data/processed/y_labels.npy')

print(f"✅ Data loaded successfully")
print(f"   Segments shape: {X.shape}")
print(f"   Labels shape:   {y.shape}")
print(f"   Unique labels:  {np.unique(y)}")



✅ Data loaded successfully
   Segments shape: (100012, 360)
   Labels shape:   (100012,)
   Unique labels:  ['A' 'L' 'N' 'R' 'V']


In [2]:
from scipy.signal import find_peaks
from scipy.stats import skew, kurtosis

def extract_features(segment):
    """
    Extract clinically meaningful features from a single heartbeat segment.
    Returns a dictionary of features.
    """
    features = {}

    # ── Time Domain Features ────────────────────────────────
    features['mean']           = np.mean(segment)
    features['std']            = np.std(segment)
    features['max_amplitude']  = np.max(segment)
    features['min_amplitude']  = np.min(segment)
    features['signal_range']   = np.max(segment) - np.min(segment)
    features['rms']            = np.sqrt(np.mean(segment**2))  # Root mean square

    # ── Shape/Morphology Features ───────────────────────────
    features['skewness']       = skew(segment)       # Asymmetry of the waveform
    features['kurtosis']       = kurtosis(segment)   # Peakedness of the waveform
    features['zero_crossings'] = np.sum(np.diff(np.sign(segment)) != 0)

    # ── Peak Features ───────────────────────────────────────
    peaks, _    = find_peaks(segment, height=0.5)
    features['num_peaks']      = len(peaks)
    features['mean_peak_height'] = np.mean(segment[peaks]) if len(peaks) > 0 else 0

    # ── Energy Features ─────────────────────────────────────
    features['signal_energy']  = np.sum(segment**2)
    features['abs_sum']        = np.sum(np.abs(segment))

    # ── Slope Features ──────────────────────────────────────
    diff = np.diff(segment)
    features['mean_slope']     = np.mean(np.abs(diff))
    features['max_slope']      = np.max(np.abs(diff))

    return features

# Test it on one segment to make sure it works
test_features = extract_features(X[0])
print(f"✅ Feature extraction working")
print(f"   Number of features per beat: {len(test_features)}")
print(f"\nSample features from first beat:")
for name, value in test_features.items():
    print(f"   {name}: {value:.4f}")

✅ Feature extraction working
   Number of features per beat: 15

Sample features from first beat:
   mean: -0.0000
   std: 1.0000
   max_amplitude: 7.3895
   min_amplitude: -1.3316
   signal_range: 8.7210
   rms: 1.0000
   skewness: 5.1000
   kurtosis: 30.9140
   zero_crossings: 34.0000
   num_peaks: 4.0000
   mean_peak_height: 2.2973
   signal_energy: 360.0000
   abs_sum: 163.7192
   mean_slope: 0.0993
   max_slope: 1.9216


In [3]:
print("Extracting features from all 100,012 beats...")
print("This will take 1-2 minutes...\n")

feature_list = []

for i, segment in enumerate(X):
    features = extract_features(segment)
    feature_list.append(features)

    # Print progress every 10,000 beats
    if (i + 1) % 10000 == 0:
        print(f"  Processed {i+1:,} / {len(X):,} beats...")

# Convert to a DataFrame
feature_df = pd.DataFrame(feature_list)
feature_df['label'] = y

print(f"\n✅ Done!")
print(f"   Feature matrix shape: {feature_df.shape}")
print(f"\nFirst 5 rows:")
feature_df.head()

Extracting features from all 100,012 beats...
This will take 1-2 minutes...

  Processed 10,000 / 100,012 beats...
  Processed 20,000 / 100,012 beats...
  Processed 30,000 / 100,012 beats...
  Processed 40,000 / 100,012 beats...
  Processed 50,000 / 100,012 beats...
  Processed 60,000 / 100,012 beats...
  Processed 70,000 / 100,012 beats...
  Processed 80,000 / 100,012 beats...
  Processed 90,000 / 100,012 beats...
  Processed 100,000 / 100,012 beats...

✅ Done!
   Feature matrix shape: (100012, 16)

First 5 rows:


,mean,std,max_amplitude,min_amplitude,signal_range,rms,skewness,kurtosis,zero_crossings,num_peaks,mean_peak_height,signal_energy,abs_sum,mean_slope,max_slope,label
0,-2.171103e-16,1.0,7.389487,-1.331561,8.721049,1.0,5.099975,30.913988,34,4,2.297282,359.999957,163.719213,0.099312,1.921587,N
1,6.908054e-17,1.0,8.416725,-1.547450,9.964175,1.0,5.974568,41.714788,32,5,2.229819,359.999953,149.647875,0.109479,3.060890,N
2,1.578984e-16,1.0,8.182303,-2.140242,10.322545,1.0,5.654252,39.148715,22,3,3.141060,359.999951,153.710466,0.113868,2.812122,N
3,-2.467162e-16,1.0,8.334857,-1.686145,10.021002,1.0,5.651994,39.482946,33,6,1.937575,359.999948,163.206059,0.118306,2.821798,N
4,2.565849e-16,1.0,8.240655,-1.492268,9.732923,1.0,5.526280,37.605972,29,6,1.984586,359.999951,164.925400,0.112805,2.654434,N


In [4]:
# Save features to CSV so we don't have to recompute
feature_df.to_csv('data/processed/features.csv', index=False)
print("✅ Features saved to data/processed/features.csv")

✅ Features saved to data/processed/features.csv
